<a href="https://colab.research.google.com/github/areyesp-77/intro_nlp/blob/main/session10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Métricas de evaluación

Task:

Evaluar el rendimiento de tres POS-taggers respecto del *gold* *standard*.

Usar métricas estándar en PLN para demostrar qué tagger tiene un rendimiento más cercano al juicio de los expertos.

Para estos ejercicios se hará uso de la librería [sckit-learn](https://scikit-learn.org/stable/).



In [ ]:
# Importar librerías
import numpy as np
import sklearn
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt


## Ejemplo de evaluación

    Sea gold el conjunto de etiquetas determinadas por los expertos para la oración S.

    Sea model el conjunto de etiquetas que propone el modelo M para la oración S.

Determinar qué tan exacto fue el modelo considerando las decisciones establecidas en el *gold* *standard*.

In [ ]:
# Gold Standard (juicios de los expertos)
gold = ['N', 'V', 'D', 'N', 'P', 'N', 'AJ', 'V', 'N', 'PR']

# Etiquetas que asigna el modelo
model = ['N', 'V', 'D', 'AJ',  'P', 'N', 'AJ', 'N', 'N', 'PR']


En este ejemplo, hay que notar que el modelo falló en dos ocasiones:
-  La primera, confundió N con A en la etiqueta 4
-  La segunda, confundió V con N en la etiqueta 8

Con base en este comportamiento, se podría decir que el modelo tiene un rendimiento de 80%, pues de 10 posibilidades, acertó en 8, pero falló en 2 (8/10 = 0.8).

Este rendimiento se debe representar con la métrica de **accuracy**, la cual muestra la proporción de predicciones correctas del total de predicciones. Se calcula a partir de la siguiente fórmula:

- $\text{Accuracy} = \frac{\text{correct classifications}}{\text{total classifications}}$ = $\frac{\text{TP}+\text{TN}}{\text{TP} + \text{TN}+\text{FP}+\text{FN}}$

- ¿Qué proporción de TP y TN identifica correctamente el modelo?"





In [ ]:
# A partir de las variables que deinimos anteriormente,
# la librería scikit determina la accuracy llamando al método accuracy_score
# sklearn.accuracy_score(y_true, y_pred, *, normalize=True, sample_weight=None)

accuracy = accuracy_score(gold, model)
                        #(y_true, y_pred)
print(('Accuracy = ' f"{accuracy * 100:.2f}%\n"))



Si bien el porcentaje de exactitud se puede considerar aceptable, no nos da información que nos permita entender en qué se está equivocando el modelo y ello afecta su posible mejora; i.e. 80% es un resultado que invisibiliza los tipos de errores que comete el modelo; además de que es poco informativo cuando se trata de datos desbalanceados (por ejemplo, datos que no tienen una proporción 50-50, que es lo que tradicionalmente pasa cuando se está en contextos reales).

Por ello, se usan otras métricas (y representaciones) que ayudan a entender con mayor profundidad las fallas del modelo.

## Nota

En el contexto de evaluación, cobran importancia las siguientes convenciones:

- Verdaderos Positivos (**TP**): aciertos en la categoría positiva

- Verdaderos Negativos (**TN**): aciertos en la categoría negativa

- Falsos Positivos (**FP**): errores donde el modelo asigna categoría positiva, pero era negativa

- Falsos Negativos (**FN**): errores donde el modelo asigna categoría negativa, pero era positiva


### En muchas tareas, los FP y FN pueden tener un costo muy alto:
- Diagnosticar a alguien con una enfermadad cuando padece otra (FP)
- Diagnosticar a alguien sano como una persona enferma (FN)

## Precision, Recall & F1

- **Precision** indica la proporción de predicciones positivas correctas. Se obtiene con la fórmula:


  - $\text{Precision} = \frac{\text{correctly classified actual positives}}{\text{everything classified as positive}}$ = $\frac{\text{TP}}{\text{TP} + \text{FP}}$

  - ¿Qué proporción de TP identifica el modelo y que son realmente TP?

- **Recall** indica la proporción de verdaderos positivos identificados del total de verdaderos positivos posibles. Se obtiene a partir de la fórmula:

  - $\text{Recall} = \frac{\text{correctly classified actual positives}}{\text{all actual positives}}$ = $\frac{\text{TP}}{\text{TP} + \text{FN}}$

  - ¿Qué proporción de TP identifica el modelo de todos los TP que existen?

- **F1** es la media entre precision y recall. Si tiende a 1, es bueno. Su fórmula es:

  - $F_1 = \frac{2 \cdot \text{TP}}{2 \cdot \text{TP} + \text{FP} + \text{FN}}$

- **Support** es el número de muestras reales por categoría.





In [ ]:
# Informe de Precision, Recall & F1
# Estas métricas se obtienen mediante el método classification_report

print("--- Reporte por categoría gramatical ---")
print(classification_report(gold, model, zero_division=0))

En este informe se representa cómo es que se comportó el modelo considerando aciertos y errores en cada categoría.

## ¿Cómo lo interpretamos?

- Precision: de las categorías que asignó el modelo a cada palabra, cuántas pertenecían a la categoría correcta; i.e. se cuantifican los TP y se evitan los FP.


- Recall: considerando todas las palabras de cada categoría, cuántas  de ellas logró identificar correctamente; i.e. se cuantifican los TP y se evitan los FN.


- F1: es la media aritmética entre los valores de Precision y Recall.

## Análisis por categoría

- Las categorías D, P y PR tuvieron valores perfectos (1). Esto significa que el modelo identificó la única palabra que había de cada categoría sin equivocarse (P) y, claro está, sin omitir ninguna (R).

- La categoría AJ tiene una P de 0.5, pero un R de 1. Esto signfica que el modelo identificó un adjetivo correcto (TP), pero también se equivocó, pues asignó AJ a una palabra cuya etiqueta era N (FP). El R es perfecto, pues identificó el único AJ que había en el corpus; i.e. no hubo FN.

- La categoría V se comportó a la inversa, tiene una P de 1, pero un R de 0.5. Ello significa que el V que identificó era, en efecto, un V (TP), pero no identificó todos los V que debía: solo 1 de 2 posibles.

- La categoría más amplia fue la de N (4), aquí el modelo tuvo 0.75 en ambas métricas; i.e. de los tres V que identificó, los tres eran V, pero de los cuatro que debía identificar, solo lo hizo en tres.

En todos los casos, la F1 muestra un promedio entre P & R que evidencia un comportamiento global.

- Macro Avg: estima el promedio de las métricas sumando cada categoría por igual, sin importar si una categoría tiene 1 palabra o 4. Es útil para ver si el modelo es bueno con las categorías minoritarias.

- Weighted Avg: estima el promedio ponderado asignando más peso a las categorías que aparecen más veces (N).



In [ ]:
# En PLN hay una forma de ver este comportamiento: matriz de confusión
# Se obtiene con el método confusion_matrix

labels = sorted(list(set(gold + model)))
cm = confusion_matrix(gold, model, labels=labels)

# Graficar la matriz de confusión
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=labels, yticklabels=labels)
plt.title('Matriz de confusión' + '\n')
plt.ylabel('Etiqueta del Gold Standard')
plt.xlabel('\n' + 'Etiqueta del modelo')
plt.show()


## Veamos otro ejemplo para profundizar en la interpretación y el análisis


    y_true = ['violencia', 'amor', 'amistad', 'amistad', 'violencia', 'violencia', 'violencia', 'amor', 'otro']

    y_pred = ['amor', 'amistad', 'amistad', 'violencia', 'amor', 'violencia', 'violencia', 'amistad', 'otro']

Determinar la exactitud del modelo, argumentar el comportamiento mediante la obtención de P, R & F1.

Hacer la matriz de confusión.

¿Qué implicaciones tiene el comportamiento del modelo?


In [ ]:
y_true = ['violencia', 'amor', 'amistad', 'amistad', 'violencia', 'violencia', 'violencia', 'amor', 'otro']
y_pred = ['amor', 'amistad', 'amistad', 'violencia', 'amor', 'violencia', 'violencia', 'amistad', 'otro']

# Accuracy
exactitud = accuracy_score(y_true, y_pred)
print('Accuracy = ' f"{exactitud * 100:.2f}%\n")

# Precision, Recall & F1-Score
print(classification_report(y_true, y_pred, zero_division=0))

# Matriz de confusión
labels = sorted(list(set(y_true + y_pred)))
cm = confusion_matrix(y_true, y_pred, labels=labels)

# Graficar matriz de confusión
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', xticklabels=labels, yticklabels=labels)
plt.title('Matriz de confusión')
plt.ylabel('Etiqueta del Gold Standard')
plt.xlabel('Etiqueta propuesta por el Modelo')
plt.tight_layout()
plt.show()

# Evaluación de POS-taggers

El comportamiento de cada tagger, así como las etiquetas del *gold* *standard* están en el archivo **[assessment.csv](https://drive.google.com/file/d/1bdKM7fo39E-ZFOgl_s6OtCD2zIdq0CtR/view?usp=drive_link)**



In [ ]:
# Cargamos el archivo con los datos

df = pd.read_csv('assessment.csv')
print(df.head(8))

## Tree Tagger

In [ ]:
# Convertir los datos en LIST

goldstd = df.Gold_STD.tolist()
treetgr = df.TreeTger.tolist()

# Accuracy
accuracy_treetgr = accuracy_score(goldstd, treetgr)
print(f"\n--- Accuracy Tree Tagger ---")
print(f"{accuracy_treetgr * 100:.2f}%\n")

# Precision, Recall & F1-Score
print("--- Results ---")
print(classification_report(goldstd, treetgr, zero_division=0))

# Matriz de confusión
labels = sorted(list(set(goldstd + treetgr)))
cm = confusion_matrix(goldstd, treetgr, labels=labels)

# Gráfico
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='inferno', xticklabels=labels, yticklabels=labels)
plt.title('Matriz de confusión')
plt.ylabel('Gold Standard')
plt.xlabel('Tree Tagger')
plt.tight_layout()
plt.show()

## spaCy

In [ ]:
# Convertir los datos en LIST

goldstd = df.Gold_STD.tolist()
spacy = df.spaCy.tolist()

# Accuracy
accuracy_spacy = accuracy_score(goldstd, spacy)
print(f"\n--- Accuracy spaCy ---")
print(f"{accuracy_spacy * 100:.2f}%\n")

# Precision, Recall & F1-Score
print("--- Results ---")
print(classification_report(goldstd, spacy, zero_division=0))

# Matriz de confusión
labels = sorted(list(set(goldstd + spacy)))
cm = confusion_matrix(goldstd, spacy, labels=labels)

# Gráfico
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='plasma', xticklabels=labels, yticklabels=labels)
plt.title('Matriz de confusión')
plt.ylabel('Gold Standard')
plt.xlabel('spaCy')
plt.tight_layout()
plt.show()

## Stanza

In [ ]:
# Convertir los datos en LIST

# ¿?

# Resultados

In [ ]:
# Accuracy por tagger

print('Tree Tagger = ' f"{accuracy_treetgr * 100:.2f}%\n")
print('Tree spaCy = ' f"{accuracy_spacy * 100:.2f}%\n")
print('Tree Stanza = ' f"{accuracy_stanza * 100:.2f}%\n")


# Más allá de la exactitud, ¿a cuál tagger le confiarían su anotación y por qué?

Para responder la pregunta, deben hacer el análisis de P, R & F1, así como revisar la matriz de confusión. Lo anterior les permitiría decidir, por ejemplo, si "prefieren" que el tagger tienda a equicarse en determinadas categorías o qué presente mayor precisión en sus decisiones.

In [ ]:
# Precision, Recall & F1

print('Tree Tagger')
print(classification_report(goldstd, treetgr, zero_division=0))
print('spaCy')
print(classification_report(goldstd, spacy, zero_division=0))
print('Stanza')
print(classification_report(goldstd, stanza, zero_division=0))

## Exploración de resultados mediante DFs y gráficos

In [ ]:

# Generar el reporte de clasificación para Tree Tagger
treetgr_dict = classification_report(goldstd, treetgr, output_dict=True, zero_division=0)
df_report_treetgr = pd.DataFrame(treetgr_dict).transpose()
df_report_treetgr = df_report_treetgr.round(2) # Redondear para mejor legibilidad
df_report_treetgr = df_report_treetgr.add_prefix('tt_')

# Generar el reporte de clasificación para spaCy
spacy_dict = classification_report(goldstd, spacy, output_dict=True, zero_division=0)
df_report_spacy = pd.DataFrame(spacy_dict).transpose()
df_report_spacy = df_report_spacy.round(2)
df_report_spacy = df_report_spacy.add_prefix('sp_')

# Generar el reporte de clasificación para Stanza
stanza_dict = classification_report(goldstd, stanza, output_dict=True, zero_division=0)
df_report_stanza = pd.DataFrame(stanza_dict).transpose()
df_report_stanza = df_report_stanza.round(2)
df_report_stanza = df_report_stanza.add_prefix('st_')

# Unir los DataFrames para una comparación de los tres taggers
df_comparison = pd.concat([df_report_treetgr, df_report_spacy, df_report_stanza], axis=1)

#display(df_comparison)


In [ ]:
from google.colab import files

# Graficar valores de Precision

# Filtrar las filas de comparison (accuracy, macro avg, weighted avg)
df_scores = df_comparison.drop(index=['macro avg', 'weighted avg', 'accuracy'])

# Seleccionar solo las columnas de Precision
df_precision = df_scores[['tt_precision', 'sp_precision', 'st_precision']]

# Reorganizar el DF
df_precision_combined = df_precision.reset_index().melt(id_vars='index', var_name='Model', value_name='Precision')
df_precision_combined['Model'] = df_precision_combined['Model'].str.replace('_precision', '')

# Plot
plt.figure(figsize=(9, 6))
sns.barplot(x='index', y='Precision', hue='Model', data=df_precision_combined, palette='YlGnBu')#bone #PuBu #ocean
plt.title('Comparación de valores de Precision')
plt.xlabel('POS')
plt.ylabel('Precision')
plt.xticks(rotation=45, ha='right') # Rotar etiquetas para mejor legibilidad
plt.ylim(0, 1)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title='POS-taggers', loc='best', bbox_to_anchor=(1, 1) )
plt.tight_layout()
plt.savefig('precision.png')
plt.show()

files.download('precision.png')

In [ ]:
# Graficar valores de Recall

# Seleccionar solo las columnas de Recall
df_recall = df_scores[['tt_recall', 'sp_recall', 'st_recall']]

# Reorganizar el DF
df_recall_combined = df_recall.reset_index().melt(id_vars='index', var_name='Model', value_name='Recall')
df_recall_combined['Model'] = df_recall_combined['Model'].str.replace('_recall', '')

# Plot
plt.figure(figsize=(9, 6))
sns.barplot(x='index', y='Recall', hue='Model', data=df_recall_combined, palette='PuBu')#bone #PuBu #ocean
plt.title('Comparación de valores de Recall')
plt.xlabel('POS')
plt.ylabel('Recall')
plt.xticks(rotation=45, ha='right') # Rotar etiquetas para mejor legibilidad
plt.ylim(0, 1)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title='POS-taggers', loc='best', bbox_to_anchor=(1, 1) )
plt.tight_layout()
plt.savefig('recall.png')
plt.show()

files.download('recall.png')

In [ ]:
# Graficar valores de F1

# Seleccionar solo las columnas de F1
df_f1 = df_scores[['tt_f1-score', 'sp_f1-score', 'st_f1-score']]

# Reorganizar el DF
df_f1_combined = df_f1.reset_index().melt(id_vars='index', var_name='Model', value_name='F1 score')
df_f1_combined['Model'] = df_f1_combined['Model'].str.replace('_f1-score', '')

# Plot
plt.figure(figsize=(9, 6))
sns.barplot(x='index', y='F1 score', hue='Model', data=df_f1_combined, palette='tab20')
plt.title('Comparación de valores de F1')
plt.xlabel('POS')
plt.ylabel('F1')
plt.xticks(rotation=45, ha='right') # Rotar etiquetas para mejor legibilidad
plt.ylim(0, 1)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.legend(title='POS-taggers', loc='best', bbox_to_anchor=(1, 1) )
plt.tight_layout()
plt.savefig('f1.png')
plt.show()

files.download('f1.png')

In [ ]:
# Comparación de rendimiento de los mejores taggers

# Selección de datos = F1
df_f1_scatter = df_f1[['sp_f1-score', 'st_f1-score']].copy()
plt.figure(figsize=(10, 8))
sns.scatterplot(x='sp_f1-score', y='st_f1-score', data=df_f1_scatter, s=100, hue=df_f1.index, palette='viridis')

# Añadir etiquetas a cada punto
for i, row in df_f1_scatter.iterrows():
    plt.text(row['sp_f1-score'] + 0.01, row['st_f1-score'] + 0.01, i, fontsize=9)

# Añadir una línea diagonal para la referencia (y=x)
max_f1 = df_f1_scatter.max().max()
plt.plot([0, max_f1], [0, max_f1], color='red', linestyle='dotted', label='Mismo comportamiento')

plt.title('Comparación de F1 entre spaCy y Stanza)')
plt.xlabel('F1 de spaCy')
plt.ylabel('F1 de Stanza')
plt.xlim(-0.05, max_f1 + 0.1) # Ajustar límites para que las etiquetas sean visibles
plt.ylim(-0.05, max_f1 + 0.1)
plt.grid(True, linestyle='--')
plt.legend(title='Categorías', loc='best', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig('rendimientoF1.png')
plt.show()


files.download('rendimientoF1.png')